## Content-based filtering

In this document, we implement a content-based filtering recommender system that suggests movies by analyzing their attributes and comparing them to a user's profile. Unlike collaborative filtering, which relies on user interactions and preferences across a broad audience, content-based filtering focuses solely on the characteristics of the movies themselves.

Our system builds a user profile based on the movies they have liked or interacted with in the past. We extract key attributes such as genres, keywords, tagline, and overview from the dataset and use them to find similarities between the user’s preferences and other movies. By leveraging text similarity techniques and genre-matching, the model recommends movies that closely align with the user's taste.

To evaluate the effectiveness of our recommendations, we apply various performance metrics like Precision, Recall, F1-score, and Text Similarity Score. These metrics help us assess how well the system captures user preferences and provides relevant movie suggestions.

### Libraries

In [10]:
import pandas as pd
import sys
import os
from pathlib import Path
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

### Setting working directory

We set the working directory to be the parent directory of this file. This way we can more effectively load other external (data) files.

In [11]:
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.append(str(PROJECT_ROOT))
import isa_project_1.config as config
from testing import recommend_movies, test_accuracy, generate_user_profile

2025-05-11 15:32:59.456 | INFO     | ML_pipeline.isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ml_backend\ML_pipeline


2025-05-11 15:32:59.495 | INFO     | isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ml_backend\ML_pipeline


### Loading dataset

In [12]:
print("opening dataset...")
# here we read the respective output file where the dataset was stored 
movie_df = pd.read_csv(config.INTERIM_DATASETS['tmdb_5000_movies.csv'])
predictors = ('genres', 'keywords', 'tagline', 'overview')
print("success!")

opening dataset...
success!


### Data Processing

First the data must be properly preprocessed. This includes following actions:

1) **Replacing NaNs**
2) **Extracting key data from JSON-like objects**
3) **Concentating the independent variables (predictors)**
4) **Futher text preprocessing (to lowercase, removing non-character letters, removing stopwords)**

#### nltk library

nltk is a useful python library that work with human language data. In the content of our project, it's going to help by **providing stopwords** that are available in English language.

In [13]:
# Check if stopwords are already downloaded
try:
    stop_words = set(stopwords.words('english'))  # Try accessing stopwords
except LookupError:
    nltk.download('stopwords')  # Download if not found
    stop_words = set(stopwords.words('english'))  # Load stopwords after download

print("Stopwords loaded successfully!")

Stopwords loaded successfully!


#### Replacing NaN values

In [14]:
row_count = len(movie_df)

for predictor in predictors:
    print(f"{predictor}: {movie_df[predictor].isna().sum() / row_count * 100:.2f}% missing values")

genres: 0.00% missing values
keywords: 0.00% missing values
tagline: 17.57% missing values
overview: 0.06% missing values


Only tagline and overview have missing values. Tagline has quite a significant amount so removing the entire rows could have
a negative effect on the model's prediction. Instead of removing them completely, we've decided to replace them with
the following value:

'[]'

This means an empty json list. As for the overview, it contains only minumum amount of NaNs. Nevertheless, we are still going to
impute a new value (empty string this time).

In [15]:
# Filling NaN values with empty strings
movie_df['genres'] = movie_df['genres'].fillna('[]')
movie_df['keywords'] = movie_df['keywords'].fillna('[]')
movie_df['tagline'] = movie_df['tagline'].fillna('')
movie_df['overview'] = movie_df['overview'].fillna('')

#### Extracting key attributes from JSON-like strings

Attributes *genres* and *keywords* which make up our predictors are stored in JSON-like lists which is not a format that can processed directly. We'are going to solve this issue by extracting only the **name key** and replacing the entire list with the collection.

For this purpose, we have declared a separate function and defined it below.

In [16]:
# Function to extract text from JSON-like columns
def extract_names(text: str, key: str):
    """Converts the provided JSON-like list into a string of one its keys delimited by whitespace.

    Args:
        text (str): the JSON list to be converted
        key (str): the element key to extract

    Returns:
        str: final string of extracted keys
    """
    try:
        # convert the string dictionary into python dictionary
        lst = ast.literal_eval(text)

        # extracting the key
        return ' '.join([i[key] for i in lst])
    except (ValueError, SyntaxError):
        return ''

Now to replace the strings we simply call the function.

In [17]:
# Apply extraction function to genres and keywords
movie_df['genres'] = movie_df['genres'].apply(extract_names, key='name')
movie_df['keywords'] = movie_df['keywords'].apply(extract_names, key='name')

print(movie_df['genres'].head(5))
print(movie_df['keywords'].head(5))


0    Action Adventure Fantasy Science Fiction
1                    Adventure Fantasy Action
2                      Action Adventure Crime
3                 Action Crime Drama Thriller
4            Action Adventure Science Fiction
Name: genres, dtype: object
0    culture clash future space war space colony so...
1    ocean drug abuse exotic island east india trad...
2    spy based on novel secret agent sequel mi6 bri...
3    dc comics crime fighter terrorist secret ident...
4    based on novel mars medallion space travel pri...
Name: keywords, dtype: object


#### Concentatining the independent variables

It's a good practice to combine all text predictors into single text by creating a separate feature *combined_text*. 

In [18]:
# combinining relevant features into single text
movie_df['combined_text'] = movie_df['overview'] + ' ' + movie_df['tagline'] + ' ' + movie_df['genres'] + ' ' + movie_df['keywords']

#### Further text preprocessing

Finally the combined text needs some additional preprocessing:
1) making the text lowercase
2) removing non-word characters (!@#$%^&, etc.)
3) removing stopwords

In [19]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str):
    """Converts text to lowercase, removes special characters and stopwords, and applies lemmatization."""
    
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)  # Remove special characters
    words = text.split()
    
    # Remove stopwords and lemmatize
    text = ' '.join([lemmatizer.lemmatize(word) for word in words if word not in stop_words])
    
    return text

movie_df['processed_text'] = movie_df['combined_text'].apply(preprocess_text)

### Feature extraction method - TF-IDF

In the data-preprocessing we have created a single attribute which combines all of the predictors. These predictors had been preprocessed to remove any unwanted phenomenom. Now it is time to create a **TF-IDF matrix**.

We use the **TfidfVectorizer** from *sklearn* to vectorize all the movies. By doing so we'll get a TF_IDF matrix containing documents (rows) and all words (columns). Every value represent measure of how important a word is to the entire corpus.

In [20]:
# TF-IDF Vectorization
# TF-IDF is a statistical measure that evaluates how important a word (term) is
# to a document or collection (corpus)

# TF-IDF = TF x IDF
# TF = occur_of_term_in_document / total_words_in_document,  how often the word appears in the document, normalized by document length.
# IDF = total_num_of_documents / num_of_documents_containing_the_term, how important the word is, based on how many documents contain it.
# Rare words (low frequency in corpus) get a higher IDF score.
#Common words (appearing in many documents) get a lower IDF score.
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movie_df['processed_text'])

#Initialize CountVectorizer with stop words
count_vectorizer = CountVectorizer(stop_words='english')

# Apply CountVectorizer on the processed text column
count_matrix = count_vectorizer.fit_transform(movie_df['processed_text'])

### Testing

After the data is successfully preprocessed, we can start testing. The first feature defined by our business goal is the user profile. In our context, the user profile is a complete history of movies that they liked or interacted with in some way.

For development purposes, we've decided to use a list of n randomly selected movies from the dataset. Note that n is represented by the USER_LIKED_MOVIES constant.

Now, in order to test the dataset, we're going to use various metrics like Precision, Recall, or F1-score. We can't use Accuracy here because it is tricky to define TN (genres that were not recommended and not liked). Since this number is enormous, TN becomes less meaningful. Instead, we define a simplified alternative:

*hit_rate = TP / TotalLikedGenres*

In the context of our model, the metrics mean the following:

1) **Precision** - how many recommended genres actually match the user’s liked genres, compared to the total number of recommended genres?

2) **Recall** - how many of the user's liked genres were successfully recommended, compared to the total liked genres?

3) **F1-score** - the harmonic mean between Precision and Recall.

4) **Text Similarity Score** - measures how similar the textual features (e.g., movie overviews, keywords, or taglines) of the recommended movies are to those of the user's liked movies. This is based on techniques like CountVectorizer, TF-IDF, or word embeddings.

By using these metrics, we evaluate how well the recommendation system aligns with the user's preferences, both in terms of genre and textual characteristics.

During testing we are going to utilize several functions defined below.

#### Choosing similarity

To calculate similarity between vectors we have two available techniques:

1) **Cosine Similarity**
2) **Euclidian Distance**

We've decided to apply the first technique as it **ignores document length** and is generally more preferred in NLP and search engines.

In [21]:
movie_df['title']

0                                         Avatar
1       Pirates of the Caribbean: At World's End
2                                        Spectre
3                          The Dark Knight Rises
4                                    John Carter
                          ...                   
4798                                 El Mariachi
4799                                   Newlyweds
4800                   Signed, Sealed, Delivered
4801                            Shanghai Calling
4802                           My Date with Drew
Name: title, Length: 4803, dtype: object

#### Scenario 1 - user_liked_movies > recommended_movies

First we are going to test the more regular case occuring in recommender system. User profile consists of movies user has already interacted with. As user interacts with the system this number grows and is generally greater than the amount of recommended movies in a single session.

In [22]:
USER_LIKED_MOVIES_COUNT = 1000
RECOMMENDED_MOVIES = 10

##### User profile

In [23]:
user_liked_movies = generate_user_profile(movie_df=movie_df, matrix=tfidf_matrix, count=USER_LIKED_MOVIES_COUNT, starter_movies=['Mission: Impossible'])
liked_movies_df = movie_df[movie_df['title'].isin(user_liked_movies)][['title', 'genres', 'overview', 'tagline', 'keywords']]

# user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()

# Print the details
print("User Profile: Liked Movies Details")
print(liked_movies_df.to_string(index=False))  # to_string() for a cleaner display

User Profile: Liked Movies Details
                                             title                                                   genres                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [24]:
recommended_movies = recommend_movies(movie_df=movie_df, count=RECOMMENDED_MOVIES, user_liked_movies=user_liked_movies, matrix=tfidf_matrix)
print("Top Recommended Movies:")

print(recommended_movies.to_string(index=False))  # to_string() for a cleaner display

# recommended_movies_list = list(recommended_movies['title'])
# recommended_movies_list

Top Recommended Movies:
              title                                   genres  similarity
Mission: Impossible                Adventure Action Thriller    0.364760
The Helix... Loaded            Action Comedy Science Fiction    0.228383
          Frequency     Crime Drama Science Fiction Thriller    0.202276
         The Iceman                     Thriller Crime Drama    0.199166
      Dead Man Down              Thriller Action Crime Drama    0.196312
            Timecop    Thriller Science Fiction Action Crime    0.194617
       Man of Steel Action Adventure Fantasy Science Fiction    0.192197
  Brooklyn's Finest                     Crime Drama Thriller    0.191795
Four Single Fathers                             Drama Comedy    0.191635
      The Dead Girl             Mystery Drama Crime Thriller    0.191164


##### Testing model's accuracy

We utilize the test_accuracy function and we calculate the model's accuracy using various metrics. 

In [25]:
accuracies = test_accuracy(movie_df=movie_df, recommended_movies=recommended_movies, vectorizer=tfidf_vectorizer, user_liked_movies=user_liked_movies)
accuracies

{'precision': '1.0000',
 'recall': '0.4545',
 'f1_score': '0.6250',
 'text_similarity_score': '0.0295'}

### Scenario 2 - user_liked_movies < recommended_movies

This scenario is more unlikely to happen. However we still test how it influence the metrics.

In [26]:
USER_LIKED_MOVIES_COUNT = 10
RECOMMENDED_MOVIES = 100

#### Recommending movies

##### User Profile

In [27]:
# user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()

user_liked_movies = generate_user_profile(movie_df=movie_df, matrix=tfidf_matrix, count=USER_LIKED_MOVIES_COUNT, starter_movies=['Mission: Impossible'])
liked_movies_df = movie_df[movie_df['title'].isin(user_liked_movies)][['title', 'genres', 'overview', 'tagline', 'keywords']]

# user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()

# Print the details
print("User Profile: Liked Movies Details")
print(liked_movies_df.to_string(index=False))  # to_string() for a cleaner display

# print("User Profile:")
# user_liked_movies

User Profile: Liked Movies Details
                     title                           genres                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              overview                                                                         tagline                                                                                                                                                                                                                    

In [28]:
# user_profile = create_user_profile_vector(USER_LIKED_MOVIES_COUNT)
# Randomly select n movies from the entire dataset
user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()
recommended_movies = recommend_movies(movie_df=movie_df, count=RECOMMENDED_MOVIES, user_liked_movies=user_liked_movies, matrix=tfidf_matrix)

# print("Interacted movies:", user_liked_movies)

print("Top Recommended Movies:")
print(recommended_movies.to_string(index=False))  # to_string() for a cleaner display

# recommended_movies_list = list(recommended_movies['title'])
# recommended_movies_list

Top Recommended Movies:
                                    title                                                           genres  similarity
              Aliens vs Predator: Requiem                   Fantasy Action Science Fiction Thriller Horror    0.144385
                      The Helix... Loaded                                    Action Comedy Science Fiction    0.137519
                         Meet the Fockers                                                   Comedy Romance    0.136830
                   Mission: Impossible II                                        Adventure Action Thriller    0.129899
                      Mission: Impossible                                        Adventure Action Thriller    0.129031
                                   Avatar                         Action Adventure Fantasy Science Fiction    0.123260
      Harry Potter and the Goblet of Fire                                         Adventure Fantasy Family    0.120651
 Harry Potter and the Ph

#### Testing model's accuracy

In [29]:
accuracies = test_accuracy(movie_df=movie_df, recommended_movies=recommended_movies, vectorizer=tfidf_vectorizer, user_liked_movies=user_liked_movies)
accuracies

{'precision': '0.9286',
 'recall': '1.0000',
 'f1_score': '0.9630',
 'text_similarity_score': '0.0304'}

### Saving TF-IDF matrix&vectorizer

Finally, we save the both the TF-IDF matrix and vectorizer to .pkl files (binary format).

In [ ]:
# Save the TF-IDF matrix and vectorizer
with open(config.MODELS_DIR / "tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open(config.MODELS_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

with open(config.MODELS_DIR / "count_matrix.pkl", "wb") as f:
    pickle.dump(count_matrix, f)

with open(config.MODELS_DIR / "count_vectorizer.pkl", "wb") as f:
    pickle.dump(count_vectorizer, f)

movie_df.to_pickle(config.MODELS_DIR / "movie_df.pkl")

: 

### Conclusion

In this document, we have successfully performed several phases of ML pipeline:

1) **Data Preprocessing**
2) **Calculating TF-IDF/Count matrix similarities**
3) **Testing**
4) **Saving model**

In the first phase it was important to process the data into a desirable format so that they can be vectorized effectively. This process included activities like **dealing with NaNs**, **stopwords**, **JSON objects in data**, and so on. The outcome was a single string resulting from concentating all predictors together.

The second phases included **vectorization**, e.g turning text into numerical values. We tested two popular methods of this area:

1) **Count Vectorization**
2) **TF-IDF Vectorization**

The outcome of this phase was a **Count/TF-IDF matrix**. 

In the last phase we performed testing of which method is more suitable for our business goal. We identified two possible scenarios:

1) recommended_movies < liked_movies (Test 1)
2) recommended_movies > liked_movies (Test 2)

At the start of each test, we needed to create a **input user profile** containing movies user has interacted with (liked_movies). We then extracted the subset of vectors (matrix rows) that match the movies from the profile. The vectors were combined together into a single vector by calculating the mean of each word. The next step was to find the **top k (recommended_movies) most similar movies** to the united vector. Here we applied **cosine similarity** technique. The last step was to **assess the model's accuracy**. We incorporated both genre-based evaluation and text similarity which provided us with a more comprehensive assessment of the model's performance. The recall was low for Test 1 which is understandble since system couldn't cover all the genres user has interacted with.

Finally, the system was stored as **.pkl file** and is now ready for deployment.